### Задание 1.

Решите задачу распознавания лиц с помощью SVM с ядром. Попробуйте различные ядра: 'poly', 'rbf', 'sigmoid'.

Подберите гиперпараметры по кросс-валидации. 

SVM с каким ядром дал лучший результат?

In [29]:
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [30]:
faces = fetch_lfw_people(min_faces_per_person=70, resize=0.4)
X, y = faces.data, faces.target
target_names = faces.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pca = PCA(n_components=150, whiten=True, random_state=42)

svc = SVC(class_weight='balanced')

pipe = Pipeline([
    ('pca', pca),
    ('svc', svc)
])

param_grid = [
    {'svc__kernel': ['rbf'], 'svc__C': [1, 10, 100], 'svc__gamma': ['scale', 0.001, 0.01]},
    {'svc__kernel': ['poly'], 'svc__C': [1, 10], 'svc__degree': [2, 3]},
    {'svc__kernel': ['sigmoid'], 'svc__C': [1, 10], 'svc__gamma': ['scale', 0.001]}
]

grid = GridSearchCV(pipe, param_grid, cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
best_params = grid.best_params_
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=target_names)

print(best_params, acc, report[:1000]) 

Fitting 3 folds for each of 17 candidates, totalling 51 fits
{'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} 0.8163934426229508                    precision    recall  f1-score   support

     Ariel Sharon       0.67      0.71      0.69        14
     Colin Powell       0.84      0.78      0.81        65
  Donald Rumsfeld       0.83      0.76      0.79        33
    George W Bush       0.87      0.92      0.89       133
Gerhard Schroeder       0.73      0.70      0.71        23
       Tony Blair       0.68      0.68      0.68        37

         accuracy                           0.82       305
        macro avg       0.77      0.76      0.76       305
     weighted avg       0.82      0.82      0.82       305



In [31]:
print("лучшее ядро:", grid.best_estimator_.get_params()['svc__kernel'])

лучшее ядро: rbf


In [32]:
print("лучший f1 (средний по кросс-валидации):", round(grid.best_score_, 4))

лучший f1 (средний по кросс-валидации): 0.8158


In [33]:
cv_results = pd.DataFrame(grid.cv_results_)
display(cv_results[['param_svc__kernel', 'param_svc__C', 'param_svc__gamma', 
                    'param_svc__degree', 'mean_test_score', 'rank_test_score']]
        .sort_values(by='mean_test_score', ascending=False).head())

,param_svc__kernel,param_svc__C,param_svc__gamma,param_svc__degree,mean_test_score,rank_test_score
4,rbf,10,0.001,NaN,0.815789,1
6,rbf,100,scale,NaN,0.814693,2
3,rbf,10,scale,NaN,0.814693,2
1,rbf,1,0.001,NaN,0.812500,4
16,sigmoid,10,0.001,NaN,0.810307,5


### Задание 2.

Решите задачу распознавания лиц с помощью логистической регрессии (она также поддерживает опцию class_weight='balanced'):

1) Объявите модель, состоящую из pipeline(pca,logistic regression)

2) Подберите по сетке параметр C логистической регрессии (с помощью GridSearch)

3) Обучите модель на тренировочных данных и выведите наилучшие параметры модели

Какое качество показала эта модель?

In [34]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

model = make_pipeline(
    PCA(n_components=150, whiten=True, random_state=42),
    lr
)

param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(model, param_grid, cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)

print("лучшие параметры:", grid.best_params_)
print("accuracy:", accuracy_score(y_test, y_pred))
print("итоги:\n", classification_report(y_test, y_pred, target_names=target_names))


Fitting 3 folds for each of 5 candidates, totalling 15 fits
лучшие параметры: {'logisticregression__C': 0.1}
accuracy: 0.8065573770491803
итоги:
                    precision    recall  f1-score   support

     Ariel Sharon       0.64      0.64      0.64        14
     Colin Powell       0.83      0.75      0.79        65
  Donald Rumsfeld       0.76      0.76      0.76        33
    George W Bush       0.89      0.88      0.89       133
Gerhard Schroeder       0.69      0.78      0.73        23
       Tony Blair       0.67      0.76      0.71        37

         accuracy                           0.81       305
        macro avg       0.75      0.76      0.75       305
     weighted avg       0.81      0.81      0.81       305



### Задание 3.

Разбалловка:

- 5 баллов: обучили один алгоритм и погридсерчили
- 10 баллов: попробовали обучить два и более алгоритмов, погридсерчили

Поработайте с датасетом winequalityN (целевая переменная - quality). Поэкспериментируйте с алгоритмами классификации, попробуйте подобрать гиперпараметры для них. 

In [35]:
df = pd.read_csv("winequalityN.csv") 

data = df.copy()

data['type'] = LabelEncoder().fit_transform(data['type'])

data_clean = data.dropna()

X = data_clean.drop('quality', axis=1)
y = data_clean['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [36]:
rf_model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

rf_param_grid = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [None, 10, 20],
    'randomforestclassifier__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(rf_model, rf_param_grid, cv=3, n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

rf_y_pred = rf_grid.predict(X_test)

rf_best_params = rf_grid.best_params_
rf_accuracy = accuracy_score(y_test, rf_y_pred)
rf_report = classification_report(y_test, rf_y_pred, zero_division=0)

#rf_best_params, rf_accuracy, rf_report[:1000]

rf_result = f"""

RandomForestClassifier

лучшие параметры:
{rf_best_params}

accuracy: {rf_accuracy:.3f}

итоги:
{rf_report}
"""

print(rf_result)


Fitting 3 folds for each of 12 candidates, totalling 36 fits


RandomForestClassifier

лучшие параметры:
{'randomforestclassifier__max_depth': 20, 'randomforestclassifier__min_samples_split': 5, 'randomforestclassifier__n_estimators': 200}

accuracy: 0.692

итоги:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.67      0.07      0.13        54
           5       0.71      0.72      0.72       532
           6       0.66      0.79      0.72       705
           7       0.76      0.59      0.67       269
           8       0.84      0.33      0.48        48
           9       0.00      0.00      0.00         1

    accuracy                           0.69      1616
   macro avg       0.52      0.36      0.39      1616
weighted avg       0.70      0.69      0.68      1616




уверенно справляется с основными классами (5–7), но слабо распознаёт редкие классы (3, 4, 8, 9)

In [37]:
gb_model = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(random_state=42)
)

gb_param_grid = {
    'gradientboostingclassifier__n_estimators': [100, 200],
    'gradientboostingclassifier__max_depth': [3, 5],
    'gradientboostingclassifier__learning_rate': [0.05, 0.1]
}

gb_grid = GridSearchCV(gb_model, gb_param_grid, cv=3, n_jobs=-1, verbose=1)
gb_grid.fit(X_train, y_train)

gb_y_pred = gb_grid.predict(X_test)
gb_best_params = gb_grid.best_params_
gb_accuracy = accuracy_score(y_test, gb_y_pred)
gb_report = classification_report(y_test, gb_y_pred)

gb_result = f"""

GradientBoostingClassifier

лучшие параметры:
{gb_best_params}

accuracy: {gb_accuracy:.3f}

отчёт:
{gb_report}
"""

print(gb_result)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


GradientBoostingClassifier

лучшие параметры:
{'gradientboostingclassifier__learning_rate': 0.1, 'gradientboostingclassifier__max_depth': 5, 'gradientboostingclassifier__n_estimators': 200}

accuracy: 0.669

отчёт:
              precision    recall  f1-score   support

           3       0.14      0.14      0.14         7
           4       0.35      0.15      0.21        54
           5       0.70      0.71      0.70       532
           6       0.66      0.74      0.70       705
           7       0.69      0.59      0.64       269
           8       0.50      0.35      0.41        48
           9       0.00      0.00      0.00         1

    accuracy                           0.67      1616
   macro avg       0.44      0.38      0.40      1616
weighted avg       0.66      0.67      0.66      1616




похож на форест — хорошо работает с основными значениями quality, но слабо — с редкими. точность чуть ниже, но общая структура метрик близка